# Quantium Virtual Internship - Task 2
## Store Trial Analysis: 77, 86 and 88
**Analyst:** Bélgica Reyes
**Date:** May 2026
**Goal:** Evaluate the performance of trial layouts and provide a recommendation based on uplift testing.
---

In [3]:
import pandas as pd
import numpy as np


data = pd.read_csv('QVI_data.csv')

In [4]:
# 1. Convert DATE column to datetime objects
data['DATE'] = pd.to_datetime(data['DATE'])

# 2. Create YEARMONTH identifier (YYYYMM format)
data['YEARMONTH'] = data['DATE'].dt.strftime('%Y%m').astype(int)

# 3. Define monthly metrics for each store
# Aggregating total sales, unique customers, and total transaction count
monthly_stats = data.groupby(['STORE_NBR', 'YEARMONTH']).agg(
    totSales=('TOT_SALES', 'sum'),
    nCustomers=('LYLTY_CARD_NBR', 'nunique'),
    nTxnPerCust=('TXN_ID', 'count')
).reset_index()

# 4. Calculate average transactions per customer
monthly_stats['nTxnPerCust'] = monthly_stats['nTxnPerCust'] / monthly_stats['nCustomers']

# 5. Filter for stores that have data for the entire 12-month period
# This ensures a fair comparison between trial and control stores
stores_with_full_data = monthly_stats.groupby('STORE_NBR')['YEARMONTH'].count()
stores_to_keep = stores_with_full_data[stores_with_full_data == 12].index
monthly_stats = monthly_stats[monthly_stats['STORE_NBR'].isin(stores_to_keep)]

# 6. Check results
print(f"Number of stores with full 12-month data: {len(stores_to_keep)}")
monthly_stats.head()

Number of stores with full 12-month data: 260


,STORE_NBR,YEARMONTH,totSales,nCustomers,nTxnPerCust
0,1,201807,206.9,49,1.061224
1,1,201808,176.1,42,1.023810
2,1,201809,278.8,59,1.050847
3,1,201810,188.1,44,1.022727
4,1,201811,192.6,46,1.021739


### Data Preparation and Pre-Trial Period Definition
To ensure a valid comparison, we need to focus on the period before the trial started (July 2018 to January 2019). We will use this "Pre-Trial" data to find control stores that behave similarly to our trial stores (77, 86, and 88) in terms of sales and customer behavior.

In [5]:
# Define the trial period and pre-trial period
# Trial period: Feb 2019 (201902) to April 2019 (201904)
# Pre-trial period: Before Feb 2019

preTrial_data = monthly_stats[monthly_stats['YEARMONTH'] < 201902]

# Check if we have the 7 months of pre-trial data (July 2018 to Jan 2019)
print(f"Pre-trial months: {preTrial_data['YEARMONTH'].unique()}")

Pre-trial months: [201807 201808 201809 201810 201811 201812 201901]


### Selection of Control Stores
To identify the best control store for each trial store (77, 86, and 88), we will calculate:
1. **Pearson Correlation:** Measures how similar the sales trends are.
2. **Magnitude Distance:** Measures how similar the absolute values are.

We will then combine these scores to find the store that is most statistically similar to each trial store during the pre-trial period.

In [6]:
def calculate_correlation(trial_store, df, metric):
    """Calculates correlation between a trial store and all other stores."""
    control_stores = df[df['STORE_NBR'] != trial_store]['STORE_NBR'].unique()
    trial_series = df[df['STORE_NBR'] == trial_store][metric].values

    correlations = []
    for store in control_stores:
        control_series = df[df['STORE_NBR'] == store][metric].values
        # Calculate correlation coefficient
        corr = np.corrcoef(trial_series, control_series)[0, 1]
        correlations.append({'Store_Control': store, 'Correlation': corr})

    return pd.DataFrame(correlations)

def calculate_magnitude_distance(trial_store, df, metric):
    """Calculates normalized distance score (0 to 1) for a given metric."""
    control_stores = df[df['STORE_NBR'] != trial_store]['STORE_NBR'].unique()
    trial_values = df[df['STORE_NBR'] == trial_store][metric].values

    distances = []
    for store in control_stores:
        control_values = df[df['STORE_NBR'] == store][metric].values
        # Mean absolute percentage error as a proxy for distance
        diff = np.abs(trial_values - control_values)
        avg_dist = np.mean(diff)

        # Normalize distance: 1 - (dist - min) / (max - min)
        distances.append({'Store_Control': store, 'Dist_Score': avg_dist})

    dist_df = pd.DataFrame(distances)
    # Normalize the distance score so 1 is closest and 0 is farthest
    dist_df['Magnitude_Score'] = 1 - (dist_df['Dist_Score'] - dist_df['Dist_Score'].min()) / (dist_df['Dist_Score'].max() - dist_df['Dist_Score'].min())

    return dist_df[['Store_Control', 'Magnitude_Score']]

### Visual Validation of Control Store Selection
To confirm that Store 233 is a suitable control for Store 77, we visualize their total sales trends during the pre-trial period. A high degree of similarity in the movement of these lines indicates that the control store accurately reflects the trial store's behavior before any changes were implemented.

In [8]:
# 1. Calculate scores
corr_sales = calculate_correlation(77, preTrial_data, 'totSales')
mag_sales = calculate_magnitude_distance(77, preTrial_data, 'totSales')
corr_cust = calculate_correlation(77, preTrial_data, 'nCustomers')
mag_cust = calculate_magnitude_distance(77, preTrial_data, 'nCustomers')

# 2. Rename columns before merging to avoid confusion
corr_sales.columns = ['Store_Control', 'Correlation_Sales']
mag_sales.columns = ['Store_Control', 'Magnitude_Score_Sales']
corr_cust.columns = ['Store_Control', 'Correlation_Cust']
mag_cust.columns = ['Store_Control', 'Magnitude_Score_Cust']

# 3. Merge all scores into one table
scores_77 = corr_sales.merge(mag_sales, on='Store_Control')
scores_77 = scores_77.merge(corr_cust, on='Store_Control')
scores_77 = scores_77.merge(mag_cust, on='Store_Control')

# 4. Final Score Calculation (Average of all 4 metrics)
scores_77['Final_Score'] = (scores_77['Correlation_Sales'] + scores_77['Magnitude_Score_Sales'] +
                           scores_77['Correlation_Cust'] + scores_77['Magnitude_Score_Cust']) / 4

# 5. Get the winner
best_match_77 = scores_77.sort_values(by='Final_Score', ascending=False).head(1)
print("Best control store for Trial Store 77:")
print(best_match_77[['Store_Control', 'Final_Score']])

Best control store for Trial Store 77:
     Store_Control  Final_Score
220            233     0.973533


### Finding Control Stores for 86 and 88
We will now repeat the correlation and magnitude distance analysis for the remaining trial stores (86 and 88) using the same pre-trial metrics.

In [9]:
# --- FINDING CONTROL FOR STORE 86 ---
# 1. Calculate scores for 86
corr_sales_86 = calculate_correlation(86, preTrial_data, 'totSales')
mag_sales_86 = calculate_magnitude_distance(86, preTrial_data, 'totSales')
corr_cust_86 = calculate_correlation(86, preTrial_data, 'nCustomers')
mag_cust_86 = calculate_magnitude_distance(86, preTrial_data, 'nCustomers')

# 2. Rename and Merge
corr_sales_86.columns = ['Store_Control', 'Correlation_Sales']
mag_sales_86.columns = ['Store_Control', 'Magnitude_Score_Sales']
corr_cust_86.columns = ['Store_Control', 'Correlation_Cust']
mag_cust_86.columns = ['Store_Control', 'Magnitude_Score_Cust']

scores_86 = corr_sales_86.merge(mag_sales_86, on='Store_Control').merge(corr_cust_86, on='Store_Control').merge(mag_cust_86, on='Store_Control')
scores_86['Final_Score'] = scores_86[['Correlation_Sales', 'Magnitude_Score_Sales', 'Correlation_Cust', 'Magnitude_Score_Cust']].mean(axis=1)

best_match_86 = scores_86.sort_values(by='Final_Score', ascending=False).head(1)

# --- FINDING CONTROL FOR STORE 88 ---
# 1. Calculate scores for 88
corr_sales_88 = calculate_correlation(88, preTrial_data, 'totSales')
mag_sales_88 = calculate_magnitude_distance(88, preTrial_data, 'totSales')
corr_cust_88 = calculate_correlation(88, preTrial_data, 'nCustomers')
mag_cust_88 = calculate_magnitude_distance(88, preTrial_data, 'nCustomers')

# 2. Rename and Merge
corr_sales_88.columns = ['Store_Control', 'Correlation_Sales']
mag_sales_88.columns = ['Store_Control', 'Magnitude_Score_Sales']
corr_cust_88.columns = ['Store_Control', 'Correlation_Cust']
mag_cust_88.columns = ['Store_Control', 'Magnitude_Score_Cust']

scores_88 = corr_sales_88.merge(mag_sales_88, on='Store_Control').merge(corr_cust_88, on='Store_Control').merge(mag_cust_88, on='Store_Control')
scores_88['Final_Score'] = scores_88[['Correlation_Sales', 'Magnitude_Score_Sales', 'Correlation_Cust', 'Magnitude_Score_Cust']].mean(axis=1)

best_match_88 = scores_88.sort_values(by='Final_Score', ascending=False).head(1)

# Print results
print("Best control for Store 86:")
print(best_match_86[['Store_Control', 'Final_Score']])
print("\nBest control for Store 88:")
print(best_match_88[['Store_Control', 'Final_Score']])

Best control for Store 86:
     Store_Control  Final_Score
146            155      0.95493

Best control for Store 88:
     Store_Control  Final_Score
224            237     0.813951


## Trial Assessment
Now that we have selected our control stores (77-233, 86-155, 88-237), we will compare their performance during the trial period (February 2019 to April 2019).
We will scale the control store's sales to match the trial store's pre-trial levels and then calculate the percentage difference to determine if there was a significant uplift.

In [10]:
# 1. Calculate the scaling factor for Sales
# Ratio of Trial Sales / Control Sales during Pre-Trial period
preTrial_77 = preTrial_data[preTrial_data['STORE_NBR'] == 77]['totSales'].sum()
preTrial_233 = preTrial_data[preTrial_data['STORE_NBR'] == 233]['totSales'].sum()
scaling_factor_sales = preTrial_77 / preTrial_233

# 2. Apply scaling to the Control Store (233) for the whole period
scaled_control_sales = monthly_stats[monthly_stats['STORE_NBR'] == 233].copy()
scaled_control_sales['controlSales'] = scaled_control_sales['totSales'] * scaling_factor_sales

# 3. Calculate Percentage Difference during the Trial Period (Feb, Mar, Apr 2019)
trial_77 = monthly_stats[monthly_stats['STORE_NBR'] == 77].copy()
trial_assessment = trial_77.merge(scaled_control_sales[['YEARMONTH', 'controlSales']], on='YEARMONTH')

# Calculate % Difference
trial_assessment['percentageDiff'] = abs(trial_assessment['totSales'] - trial_assessment['controlSales']) / trial_assessment['controlSales']

# 4. Filter only the trial months to see the impact
trial_months = [201902, 201903, 201904]
uplift_77 = trial_assessment[trial_assessment['YEARMONTH'].isin(trial_months)]

print("Trial Assessment for Store 77 (vs Control 233):")
print(uplift_77[['YEARMONTH', 'totSales', 'controlSales', 'percentageDiff']])

Trial Assessment for Store 77 (vs Control 233):
   YEARMONTH  totSales  controlSales  percentageDiff
7     201902     235.0    249.762622        0.059107
8     201903     278.5    203.802205        0.366521
9     201904     263.5    162.345704        0.623080


### Analysis of Results for Store 77
The results show a significant increase in sales for Store 77 compared to its control (Store 233), especially in March (+36.6%) and April (+62.3%). This suggests that the trial had a positive impact on total sales performance.

In [11]:
# --- TRIAL ASSESSMENT FOR STORE 86 (vs Control 155) ---

# 1. Calculate the scaling factor for Sales
preTrial_86 = preTrial_data[preTrial_data['STORE_NBR'] == 86]['totSales'].sum()
preTrial_155 = preTrial_data[preTrial_data['STORE_NBR'] == 155]['totSales'].sum()
scaling_factor_86 = preTrial_86 / preTrial_155

# 2. Apply scaling to the Control Store (155)
scaled_control_86 = monthly_stats[monthly_stats['STORE_NBR'] == 155].copy()
scaled_control_86['controlSales'] = scaled_control_86['totSales'] * scaling_factor_86

# 3. Merge and calculate Percentage Difference
trial_86 = monthly_stats[monthly_stats['STORE_NBR'] == 86].copy()
assessment_86 = trial_86.merge(scaled_control_86[['YEARMONTH', 'controlSales']], on='YEARMONTH')
assessment_86['percentageDiff'] = abs(assessment_86['totSales'] - assessment_86['controlSales']) / assessment_86['controlSales']

# 4. Filter trial months
uplift_86 = assessment_86[assessment_86['YEARMONTH'].isin([201902, 201903, 201904])]

print("Trial Assessment for Store 86 (vs Control 155):")
print(uplift_86[['YEARMONTH', 'totSales', 'controlSales', 'percentageDiff']])

Trial Assessment for Store 86 (vs Control 155):
   YEARMONTH  totSales  controlSales  percentageDiff
7     201902     913.2    864.522060        0.056306
8     201903    1026.8    780.320405        0.315870
9     201904     848.2    819.317024        0.035253


### Analysis of Results for Store 88
Finally, we evaluate Store 88 against its control, Store 237. By calculating the percentage difference in total sales during the trial period, we can conclude if the store trial was successful across all three selected locations.

In [12]:
# --- TRIAL ASSESSMENT FOR STORE 88 (vs Control 237) ---

# 1. Calculate the scaling factor for Sales
preTrial_88 = preTrial_data[preTrial_data['STORE_NBR'] == 88]['totSales'].sum()
preTrial_237 = preTrial_data[preTrial_data['STORE_NBR'] == 237]['totSales'].sum()
scaling_factor_88 = preTrial_88 / preTrial_237

# 2. Apply scaling to the Control Store (237)
scaled_control_88 = monthly_stats[monthly_stats['STORE_NBR'] == 237].copy()
scaled_control_88['controlSales'] = scaled_control_88['totSales'] * scaling_factor_88

# 3. Merge and calculate Percentage Difference
trial_88 = monthly_stats[monthly_stats['STORE_NBR'] == 88].copy()
assessment_88 = trial_88.merge(scaled_control_88[['YEARMONTH', 'controlSales']], on='YEARMONTH')
assessment_88['percentageDiff'] = abs(assessment_88['totSales'] - assessment_88['controlSales']) / assessment_88['controlSales']

# 4. Filter trial months
uplift_88 = assessment_88[assessment_88['YEARMONTH'].isin([201902, 201903, 201904])]

print("Trial Assessment for Store 88 (vs Control 237):")
print(uplift_88[['YEARMONTH', 'totSales', 'controlSales', 'percentageDiff']])

Trial Assessment for Store 88 (vs Control 237):
   YEARMONTH  totSales  controlSales  percentageDiff
7     201902    1370.2   1406.989143        0.026147
8     201903    1477.2   1210.082775        0.220743
9     201904    1439.4   1206.477165        0.193060


## Final Conclusion
The trial conducted between February 2019 and April 2019 has been a statistical success.

1. **Store 77** showed the highest uplift, reaching a **62.3%** increase in sales by April.
2. **Store 86** performed strongly, especially in March with a **31.5%** increase.
3. **Store 88** maintained a consistent uplift of approximately **20%** during the last two months of the trial.

Overall, the trial stores significantly outperformed their respective control stores. We recommend rolling out the new layout/design to other stores in the network, as it has proven to drive a clear increase in total sales.